# Barter-RS: Multi-Exchange Data Streams

This notebook demonstrates how to combine market data from multiple exchanges and
multiple subscription types into a single unified stream.

## Topics Covered
- Combining PublicTrades, OrderBooksL1, and OrderBooksL2 into one stream
- Streaming from Binance Spot, Binance Futures, and OKX simultaneously
- Using the `DataKind` enum for heterogeneous event handling
- Selecting individual exchange streams

In [ ]:
:dep barter-data = { version = "0.11" }
:dep barter-instrument = { version = "0.3" }
:dep tokio = { version = "1", features = ["full"] }
:dep tokio-stream = "0.1"
:dep futures-util = "0.3"
:dep tracing = "0.1"
:dep tracing-subscriber = { version = "0.3", features = ["env-filter"] }

## 1. Building Multi-Kind, Multi-Exchange Streams

The `Streams::builder_multi()` API composes multiple subscription-type builders
into a single `Streams<MarketStreamResult<_, DataKind>>`. This normalises all
event types into the `DataKind` enum, making it easy to handle heterogeneous data
in one event loop.

In [ ]:
use barter_data::{
    event::DataKind,
    exchange::{
        binance::{futures::BinanceFuturesUsd, spot::BinanceSpot},
        okx::Okx,
    },
    streams::{Streams, consumer::MarketStreamResult, reconnect::stream::ReconnectingStream},
    subscription::{
        book::{OrderBooksL1, OrderBooksL2},
        trade::PublicTrades,
    },
};
use barter_instrument::instrument::market_data::{
    MarketDataInstrument, kind::MarketDataInstrumentKind,
};
use tokio_stream::StreamExt;

// Build a combined stream across exchanges and data kinds
let streams: Streams<MarketStreamResult<MarketDataInstrument, DataKind>> = Streams::builder_multi()
    // PublicTrades from multiple exchanges
    .add(Streams::<PublicTrades>::builder()
        .subscribe([
            (BinanceSpot::default(), "btc", "usdt", MarketDataInstrumentKind::Spot, PublicTrades),
        ])
        .subscribe([
            (BinanceFuturesUsd::default(), "btc", "usdt", MarketDataInstrumentKind::Perpetual, PublicTrades),
        ])
        .subscribe([
            (Okx, "btc", "usdt", MarketDataInstrumentKind::Spot, PublicTrades),
        ])
    )
    // OrderBooksL1 from Binance
    .add(Streams::<OrderBooksL1>::builder()
        .subscribe([
            (BinanceSpot::default(), "btc", "usdt", MarketDataInstrumentKind::Spot, OrderBooksL1),
        ])
    )
    .init()
    .await
    .expect("failed to init multi-exchange streams");

println!("Multi-exchange streams initialised!");
println!("Reading first 10 events across all exchanges and data types...\n");

## 2. Consuming the Unified Stream

All events are normalised into `MarketEvent<_, DataKind>`. Pattern-match on
`DataKind` to handle each subscription type.

In [ ]:
let mut joined = streams
    .select_all()
    .with_error_handler(|error| eprintln!("Stream error: {error:?}"));

let mut count = 0;
while let Some(event) = joined.next().await {
    match &event.kind {
        DataKind::Trade(trade) => {
            println!(
                "[TRADE]  {:>15} {:>20} | side={:>4} price={:<12} qty={}",
                event.exchange, event.instrument,
                format!("{}", trade.side), trade.price, trade.quantity
            );
        }
        DataKind::OrderBookL1(book) => {
            println!(
                "[L1]     {:>15} {:>20} | bid={:<12} ask={}",
                event.exchange, event.instrument,
                book.best_bid.price, book.best_ask.price
            );
        }
        other => {
            println!(
                "[OTHER]  {:>15} {:>20} | kind={}",
                event.exchange, event.instrument, other.kind_name()
            );
        }
    }

    count += 1;
    if count >= 10 { break; }
}

println!("\nDone! Consumed {count} events from multiple exchanges.");

## 3. Selecting Individual Exchange Streams

Instead of `select_all()`, you can select a specific exchange's stream using
`streams.select(ExchangeId)`. This is useful when you need exchange-specific
processing pipelines.

In [ ]:
// Example: selecting only BinanceSpot events
// let binance_stream = streams.select(ExchangeId::BinanceSpot);
//
// This returns only events from the BinanceSpot exchange,
// allowing exchange-specific processing, rate limiting, or routing.

println!("Tip: Use streams.select(ExchangeId::BinanceSpot) to get a single exchange's stream.");
println!("Tip: Use streams.select_all() to merge all exchanges into one stream.");

## Supported Exchanges

| Exchange | Spot | Perpetual | Futures | Options |
|----------|------|-----------|---------|---------|
| Binance | `BinanceSpot` | `BinanceFuturesUsd` | — | — |
| OKX | `Okx` | `Okx` | — | `Okx` |
| Coinbase | `Coinbase` | — | — | — |
| Kraken | `Kraken` | — | — | — |
| Bybit | — | `BybitPerpetualsUsd` | — | — |
| Bitfinex | `Bitfinex` | — | — | — |
| Gateio | `GateioSpot` | `GateioPerpetualsUsd` | `GateioFuturesUsd` | `GateioOptions` |
| Bitmex | — | `Bitmex` | — | — |

### DataKind Variants

```rust
pub enum DataKind {
    Trade(PublicTrade),
    OrderBookL1(OrderBookL1),
    OrderBookL2(OrderBookL2),
    // ... more as needed
}
```